# Day 3 - Conversational AI - aka Chatbot!

In [2]:
# Import necessary libraries

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [3]:
# Load environment variables from .env file
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print("OpenAI API key loaded successfully.")
else:
    print("Failed to load OpenAI API key. Please check your .env file.")

OpenAI API key loaded successfully.


In [4]:
# Initialize OpenAI client
openai = OpenAI()
MODEL = "gpt-4.1-mini"

In [3]:
system_message = "You are a helpful assistant"

In [ ]:
# Define the chat function
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role":"system", "content":system_message}] + history + [{"role":"user", "content":message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [9]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [7]:
system_message = "You are a helpful assistant at a fast-food restaurant. You should suggest combo meals \
to customers to increase value. Combo meals come with a drink and fries at a discounted price. \
For example, if a customer says 'I want a burger', you could reply: \
'Sure! Would you like to make it a combo? It comes with fries and a drink for a better price.' \
Encourage customers to upgrade to combos when possible."

In [8]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [9]:
system_message += "\nIf the customer asks for pizza, you should respond that pizzas are not on sale today, \
but remind the customer to look at fries!"

In [10]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [11]:

# Now let's say we want to add a new instruction about belts, but we don't want to lose the previous instructions about pizza and combos. We can modify the chat function to include the new instruction without overwriting the old ones.
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'pizza' in message.lower():
        relevant_system_message += " The store does not sell pizza; if you are asked for pizza, be sure to point out other items on sale."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [12]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
